# 1D ADI MMS with nonzero source

Method of manufactured solutions for the 1D wave equation on a Yee/ADI grid with homogeneous Dirichlet (PEC) boundaries, comparing E-source placements A–D.

Manufactured solution (nonzero source, $\omega\neq ck$):

$$
E_{\mathrm{MMS}}(x,t)=\sin(kx)\cos(\omega t),\qquad
k=\frac{2\pi}{L},\qquad
\omega=\frac{3\pi c}{L}.
$$

Then

$$
E_{tt}-c^2 E_{xx}=f(x,t)=-\frac{5\pi^2 c^2}{L^2}\sin(kx)\cos(\omega t).
$$

In the ADI update, $\partial_t E=\varepsilon^{-1}\partial_x H+S$ with $S_t=f$, so

$$
S(x,t)=\frac{c^2k^2-\omega^2}{\omega}\sin(kx)\sin(\omega t).
$$

ICs from the MMS: $E(x,0)=\sin(kx)$, $H\equiv0$ (since $E_t(x,0)=0$ and $S(x,0)=0$).

| | placement ($S_E$ increments; draft `algo.adi_e_excitation`) |
|---|---|
| A | $\Delta t\,S^{n+1/2}$ in the first half only |
| B | $\Delta t\,S^{n+1/2}$ in the second half only |
| C | $\tfrac12\Delta t\,S^{n+1/2}$ in both halves |
| D | $\tfrac12\Delta t\,S^{n+1/4}$ then $\tfrac12\Delta t\,S^{n+3/4}$ |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

C0 = 299792458.0
EPS0 = 8.8541878128e-12
MU0 = 1.25663706212e-6

L = 8.0e-6
NZ = 2928  # match source_injection_1d
N_PERIODS = 4.0
E0 = 1.0
# ω = (3/2) c k ≠ c k  ⇒  nonzero MMS source
CFLS = [32.0, 64.0, 128.0, 256.0, 512.0]  # match source_injection_1d
SCHEMES = ("a", "b", "c", "d")

dx = L / NZ
k = 2.0 * math.pi / L
omega = 1.5 * C0 * k  # = 3 π c / L
kh = k * dx
f0 = omega / (2.0 * math.pi)
T_P = 1.0 / f0
# f = (c²k² - ω²) sin(kx) cos(ωt);  S_t = f  ⇒  S ∝ sin(ωt)
S_amp = (C0**2 * k**2 - omega**2) / omega

x_E = np.linspace(0.0, L, NZ + 1)
x_H = (np.arange(NZ) + 0.5) * dx
probe = int(np.argmin(np.abs(x_E - 0.25 * L)))

print(f"Nz={NZ}, k dx={kh:.4e}, f0={f0/1e12:.4f} THz")
print(f"omega/(c k)={omega/(C0*k):.3f}, S_amp={S_amp:.6e}")
print("CFL:", CFLS)

In [ ]:
def thomas(a, b, c, d):
    n = len(b)
    bc = np.array(b, float)
    dc = np.array(d, float)
    cc = np.array(c, float)
    for i in range(1, n):
        w = a[i - 1] / bc[i - 1]
        bc[i] -= w * cc[i - 1]
        dc[i] -= w * dc[i - 1]
    x = np.empty(n)
    x[-1] = dc[-1] / bc[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (dc[i] - cc[i] * x[i + 1]) / bc[i]
    return x


def cyclic_thomas(a, b, c, d):
    n = len(d)
    gamma = -b[0]
    b1 = b.copy()
    b1[0] -= gamma
    b1[-1] -= a[0] * c[-1] / gamma
    u = np.zeros(n)
    u[0] = gamma
    u[-1] = c[-1]
    y = thomas(a[1:], b1, c[:-1], d)
    z = thomas(a[1:], b1, c[:-1], u)
    vn = a[0] / gamma
    fact = (y[0] + vn * y[-1]) / (1.0 + z[0] + vn * z[-1])
    return y - fact * z


def implicit_lhs_coeffs(n, gamma, periodic):
    off = gamma / dx**2
    a = np.full(n, -off)
    b = np.full(n, 1.0 + 2.0 * off)
    c = np.full(n, -off)
    if periodic:
        return a, b, c
    return a[1:], b, c[:-1]


def solve_E(rhs_interior, gamma, periodic):
    n = rhs_interior.size
    if periodic:
        a, b, c = implicit_lhs_coeffs(n, gamma, True)
        return cyclic_thomas(a, b, c, rhs_interior)
    a, b, c = implicit_lhs_coeffs(n, gamma, False)
    return thomas(a, b, c, rhs_interior)


def delta_H(H, periodic):
    if periodic:
        return (H - np.roll(H, 1)) / dx
    dH = np.zeros(H.size + 1)
    dH[1:-1] = (H[1:] - H[:-1]) / dx
    return dH


def delta_E(E, periodic):
    if periodic:
        return (np.roll(E, -1) - E) / dx
    return (E[1:] - E[:-1]) / dx


def adi_step(E, H, S1, S2, C_b, D_b, gamma, periodic):
    rhs = E + C_b * delta_H(H, periodic) + S1
    if periodic:
        E_half = solve_E(rhs, gamma, True)
    else:
        E_half = np.zeros_like(E)
        E_half[1:-1] = solve_E(rhs[1:-1], gamma, False)
    dE = delta_E(E_half, periodic)
    H_half = H + D_b * dE
    E_new = E_half + C_b * delta_H(H_half, periodic) + S2
    if not periodic:
        E_new[0] = 0.0
        E_new[-1] = 0.0
    H_new = H_half + D_b * dE
    return E_new, H_new

In [ ]:
def coeffs(cfl):
    dt = cfl * dx / C0
    nsteps = int(math.ceil(N_PERIODS / (f0 * dt)))
    C_b = dt / (2.0 * EPS0)
    D_b = dt / (2.0 * MU0)
    gamma = C_b * D_b
    return dt, nsteps, C_b, D_b, gamma


def E_mms(x, t):
    return E0 * np.sin(k * x) * np.cos(omega * t)


def H_mms(x, t):
    # From H_t = μ^{-1} ∂_x E with E = E0 sin(kx) cos(ωt)
    return (E0 * k / (MU0 * omega)) * np.cos(k * x) * np.sin(omega * t)


def S_field(x, t):
    """Continuous E-equation source with S_t = f = (c²k²-ω²) sin(kx) cos(ωt)."""
    return S_amp * np.sin(k * x) * np.sin(omega * t)


def S_E(x, t, dt):
    """Full-step E increment Δt·S (split across halves by scheme)."""
    return dt * S_field(x, t)


def sources(scheme, t_n, dt, x):
    z = np.zeros_like(x)
    if scheme == "a":
        return S_E(x, t_n + 0.5 * dt, dt), z
    if scheme == "b":
        return z, S_E(x, t_n + 0.5 * dt, dt)
    if scheme == "c":
        s = 0.5 * S_E(x, t_n + 0.5 * dt, dt)
        return s, s
    if scheme == "d":
        return (
            0.5 * S_E(x, t_n + 0.25 * dt, dt),
            0.5 * S_E(x, t_n + 0.75 * dt, dt),
        )
    raise ValueError(scheme)


def l2_error(E, t):
    err = E - E_mms(x_E, t)
    return float(np.sqrt(dx * np.sum(err**2)))


def run_mms(cfl, scheme):
    dt, nsteps, C_b, D_b, gamma = coeffs(cfl)
    E = E_mms(x_E, 0.0)
    H = H_mms(x_H, 0.0)  # = 0
    t = np.empty(nsteps + 1)
    e_probe = np.empty(nsteps + 1)
    err_l2 = np.empty(nsteps + 1)
    t[0] = 0.0
    e_probe[0] = E[probe]
    err_l2[0] = l2_error(E, 0.0)
    for n in range(nsteps):
        S1, S2 = sources(scheme, n * dt, dt, x_E)
        E, H = adi_step(E, H, S1, S2, C_b, D_b, gamma, False)
        t[n + 1] = (n + 1) * dt
        e_probe[n + 1] = E[probe]
        err_l2[n + 1] = l2_error(E, t[n + 1])
    return {
        "t": t,
        "e_probe": e_probe,
        "err_l2": err_l2,
        "E": E,
        "dt": dt,
        "nsteps": nsteps,
        "err_final": err_l2[-1],
        "err_max": float(np.max(err_l2)),
        "err_probe_final": float(abs(E[probe] - E_mms(x_E[probe], t[-1]))),
    }

In [ ]:
FFT_PAD = 8

STYLES = {
    "a": ("C0", "x", r"A: $\Delta t\,S^{n+1/2}$ first half"),
    "b": ("C1", "o", r"B: $\Delta t\,S^{n+1/2}$ second half"),
    "c": ("C2", "s", r"C: $\frac{1}{2}\Delta t\,S^{n+1/2}$ both"),
    "d": ("C3", "D", r"D: $\frac{1}{2}\Delta t\,S^{n+1/4},\,\frac{1}{2}\Delta t\,S^{n+3/4}$"),
}


def last_half(times, signal):
    i0 = len(times) // 2
    return times[i0:], signal[i0:]


def compute_fft(times, signal, pad_factor=FFT_PAD):
    t, y = last_half(times, signal)
    y = y - np.mean(y)
    dt = float(np.median(np.diff(t)))
    n = len(y)
    n_fft = max(n, int(pad_factor) * n)
    spec = np.fft.rfft(y * np.hanning(n), n=n_fft)
    freqs = np.fft.rfftfreq(n_fft, d=dt)
    return freqs, np.abs(spec) / n


def peak_metrics(times, signal):
    freqs, amp = compute_fft(times, signal)
    i = 1 + int(np.argmax(amp[1:]))
    return float(amp[i]), float(freqs[i] / f0)


def analytical_f_over_f0(cfl):
    """ADI free frequency of mode k, normalized by manufactured f0 = ω/(2π)."""
    # ω_ADI/(c k) from dispersion, then ÷ (ω/(c k)) = 3/2
    return (2.0 / 1.5) * math.atan(cfl * math.sin(0.5 * kh)) / (cfl * kh)

In [ ]:
results = {scheme: {} for scheme in SCHEMES}
mms_metrics = {scheme: {} for scheme in SCHEMES}

print("=== MMS PEC (nonzero source) ===")
print(f"{'scheme':>6} {'CFL':>8} {'L2 final':>12} {'L2 max':>12} {'FFT peak':>12} {'f/f0':>10}")
for scheme in SCHEMES:
    for cfl in CFLS:
        out = run_mms(cfl, scheme)
        results[scheme][cfl] = out
        peak, ff0 = peak_metrics(out["t"], out["e_probe"])
        mms_metrics[scheme][cfl] = {"fft_peak": peak, "f_over_f0": ff0}
        print(
            f"{scheme:>6} {cfl:8g} {out['err_final']:12.4e} "
            f"{out['err_max']:12.4e} {peak:12.4e} {ff0:10.6f}"
        )

In [ ]:
# Probe time history vs MMS at a representative CFL
cfl_show = CFLS[len(CFLS)//2-1]
kh_label = f"{kh:.3e}".replace("e-0", "e-")

fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex=True)
for ax, scheme in zip(axes.ravel(), SCHEMES):
    out = results[scheme][cfl_show]
    t, ep = out["t"], out["e_probe"]
    exact = E_mms(x_E[probe], t)
    color, marker, label = STYLES[scheme]
    ax.plot(t * f0, ep, color=color, lw=1.4, label="ADI")
    ax.plot(t * f0, exact, "k--", lw=1.2, alpha=0.8, label="MMS")
    ax.set_title(f"{label}  (CFL={cfl_show:g})")
    ax.set_ylabel(r"$E(x=L/4)$")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)

for ax in axes[1]:
    ax.set_xlabel(r"$t\,f_0$")
fig.suptitle(
    rf"MMS probe: $E=\sin(kx)\cos(\omega t)$, $\omega=\frac{{3}}{{2}} ck$, $k\Delta x={kh_label}$",
    fontsize=12,
)
fig.tight_layout()
plt.show()

In [ ]:
# L2 error vs time for each scheme at fixed CFL
fig, ax = plt.subplots(figsize=(8.0, 4.8))
for scheme, (color, marker, label) in STYLES.items():
    out = results[scheme][cfl_show]
    ax.semilogy(out["t"] * f0, out["err_l2"], color=color, lw=1.5, label=label)
ax.set_xlabel(r"$t\,f_0$")
ax.set_ylabel(r"$\|E-E_{\mathrm{MMS}}\|_{L^2}$")
ax.set_title(rf"L2 error vs time (CFL={cfl_show:g}, $N_z={NZ}$)")
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Error vs CFL: final L2 and max-in-time L2
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))
x = np.asarray(CFLS)

for scheme, (color, marker, label) in STYLES.items():
    axes[0].loglog(
        x,
        [results[scheme][c]["err_final"] for c in CFLS],
        marker + "-",
        color=color,
        ms=7,
        label=label,
    )
    axes[1].loglog(
        x,
        [results[scheme][c]["err_max"] for c in CFLS],
        marker + "-",
        color=color,
        ms=7,
        label=label,
    )

# Reference slopes ~ Δt² ~ CFL² (fixed dx)
s_ref = np.array([CFLS[0], CFLS[-1]])
e0 = results["c"][CFLS[0]]["err_final"]
axes[0].loglog(s_ref, e0 * (s_ref / CFLS[0]) ** 2, "k--", lw=1.3, label=r"$\propto\mathrm{CFL}^2$")
e0m = results["c"][CFLS[0]]["err_max"]
axes[1].loglog(s_ref, e0m * (s_ref / CFLS[0]) ** 2, "k--", lw=1.3, label=r"$\propto\mathrm{CFL}^2$")

axes[0].set_xlabel("CFL $S$")
axes[0].set_ylabel(r"$\|E-E_{\mathrm{MMS}}\|_{L^2}$ at $t_{\mathrm{end}}$")
axes[0].set_title("Final-time L2 error")
axes[0].grid(alpha=0.25, which="both")
axes[0].legend(frameon=False, fontsize=7)

axes[1].set_xlabel("CFL $S$")
axes[1].set_ylabel(r"$\max_t\|E-E_{\mathrm{MMS}}\|_{L^2}$")
axes[1].set_title("Max-in-time L2 error")
axes[1].grid(alpha=0.25, which="both")
axes[1].legend(frameon=False, fontsize=7)

fig.suptitle(
    rf"MMS source schemes A–D ($N_z={NZ}$, $k\Delta x={kh_label}$, "
    rf"$f=-\frac{{5\pi^2 c^2}}{{L^2}}\sin(kx)\cos(\omega t)$)",
    fontsize=11,
)
fig.tight_layout()
plt.show()

print("\n=== CFL sensitivity (max/min final L2) ===")
for scheme in SCHEMES:
    errs = [results[scheme][c]["err_final"] for c in CFLS]
    print(f"scheme {scheme}  L2 ratio={max(errs)/min(errs):.3g}")

In [ ]:
# Spectral peak & frequency vs CFL (same layout as source_injection_1d; no theory curves)
kh_label = f"{kh:.3e}".replace("e-0", "e-")
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
x = np.asarray(CFLS)

for scheme, (color, marker, label) in STYLES.items():
    r = mms_metrics[scheme]
    axes[0].loglog(x, [r[c]["fft_peak"] for c in CFLS], marker + "-", color=color, ms=7, label=label)
    axes[1].semilogx(x, [r[c]["f_over_f0"] for c in CFLS], marker + "-", color=color, ms=7, label=label)

axes[0].set_xlabel("CFL $S$")
axes[0].set_ylabel(r"$|\mathrm{FFT}|/N$ peak")
axes[0].set_title("Spectral peak magnitude")
axes[0].grid(alpha=0.25, which="both")
axes[0].legend(frameon=False, fontsize=7)

axes[1].set_xlabel("CFL $S$")
axes[1].set_ylabel(r"$f_\mathrm{peak}/f_0$")
axes[1].set_title("Frequency (dispersion)")
axes[1].grid(alpha=0.25, which="both")
axes[1].legend(frameon=False, fontsize=8)

fig.suptitle(rf"MMS, ADI E source A–D ($k\Delta x={kh_label}$)", fontsize=12)
fig.tight_layout()
plt.show()

print("\n=== CFL sensitivity (max/min FFT peak) ===")
for scheme in SCHEMES:
    peaks = [mms_metrics[scheme][c]["fft_peak"] for c in CFLS]
    freqs = [mms_metrics[scheme][c]["f_over_f0"] for c in CFLS]
    print(
        f"mms {scheme}  FFT ratio={max(peaks)/min(peaks):.3g}  "
        f"f/f0 spread={max(freqs)-min(freqs):.4f}"
    )

In [ ]:
# Final spatial profile vs MMS + relative scheme ranking at each CFL
cfl_prof = CFLS[0]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

ax = axes[0]
exact = E_mms(x_E, results["c"][cfl_prof]["t"][-1])
ax.plot(x_E / L, exact, "k-", lw=2.0, label="MMS")
for scheme, (color, marker, label) in STYLES.items():
    E = results[scheme][cfl_prof]["E"]
    ax.plot(x_E / L, E, marker, color=color, ms=3, markevery=max(1, NZ // 32), lw=1.0, label=label)
ax.set_xlabel(r"$x/L$")
ax.set_ylabel(r"$E(x,t_{\mathrm{end}})$")
ax.set_title(rf"Profile at $t_{{\mathrm{{end}}}}$ (CFL={cfl_prof:g})")
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=7)

ax = axes[1]
for scheme, (color, marker, label) in STYLES.items():
    ax.semilogx(
        CFLS,
        [results[scheme][c]["err_final"] / results["c"][c]["err_final"] for c in CFLS],
        marker + "-",
        color=color,
        ms=7,
        label=label,
    )
ax.axhline(1.0, color="0.5", ls=":", lw=1)
ax.set_xlabel("CFL $S$")
ax.set_ylabel(r"final $L^2$ error / scheme C")
ax.set_title("Relative to symmetric midpoint (C)")
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=False, fontsize=7)

fig.suptitle(
    rf"MMS nonzero source: $k=2\pi/L$, $\omega=3\pi c/L$ ($k\Delta x={kh_label}$)",
    fontsize=11,
)
fig.tight_layout()
plt.show()


In [ ]:
# Analytical modal amplitude a(t) from draft (λ ≈ -k² ⇒ q = γ k²)
def omega_adi(cfl):
    dt = cfl * dx / C0
    return (2.0 / dt) * math.atan(cfl * math.sin(0.5 * kh))


def q_approx(cfl):
    dt = cfl * dx / C0
    gamma = (C0 * dt / 2.0) ** 2
    return gamma * k**2  # λ ≈ -k²


def a_mid2(t, cfl):
    dt = cfl * dx / C0
    w1 = omega_adi(cfl)
    th = omega * dt
    sinc = 1.0 if abs(th) < 1e-14 else math.sin(0.5 * th) / (0.5 * th)
    Ap = sinc * (C0**2 * k**2 - omega**2) / (w1**2 - omega**2)
    return (1.0 - Ap) * np.cos(w1 * t) + Ap * np.cos(omega * t)


def a_mid1(t, cfl):
    dt = cfl * dx / C0
    w1 = omega_adi(cfl)
    th = omega * dt
    qq = q_approx(cfl)
    denom = w1**2 - omega**2
    pref = 2.0 * (C0**2 * k**2 - omega**2) / (omega * dt)
    A = pref * math.sin(0.5 * th)
    B = -pref * qq * math.cos(0.5 * th)
    return (
        (1.0 - A / denom) * np.cos(w1 * t)
        - (omega / w1) * (B / denom) * np.sin(w1 * t)
        + (A / denom) * np.cos(omega * t)
        + (B / denom) * np.sin(omega * t)
    )


def a_quarter(t, cfl):
    dt = cfl * dx / C0
    w1 = omega_adi(cfl)
    th = omega * dt
    qq = q_approx(cfl)
    Ap = (
        (C0**2 * k**2 - omega**2)
        / ((w1**2 - omega**2) * omega * dt)
        * ((1.0 - qq) * math.sin(0.25 * th) + (1.0 + qq) * math.sin(0.75 * th))
    )
    return (1.0 - Ap) * np.cos(w1 * t) + Ap * np.cos(omega * t)


cfls_show = [64.0, 128.0, 256.0, 512.0]
t_plot = np.linspace(0.0, N_PERIODS / f0, 2000)

fig, axes = plt.subplots(2, 2, figsize=(11.0, 7.0), sharex=True, sharey=True)
for ax, cfl in zip(axes.ravel(), cfls_show):
    ax.plot(t_plot * f0, np.cos(omega * t_plot), "k--", lw=1.2, alpha=0.7, label=r"MMS $\cos(\omega t)$")
    ax.plot(t_plot * f0, a_mid2(t_plot, cfl), color="C2", lw=1.4, label="mid,2")
    ax.plot(t_plot * f0, a_quarter(t_plot, cfl), color="C3", lw=1.4, label="quarter")
    ax.plot(t_plot * f0, a_mid1(t_plot, cfl), color="C0", lw=1.4, label="mid,1")
    ax.set_title(
        rf"CFL$={cfl:g}$, $q\approx{q_approx(cfl):.3f}$, "
        rf"$\omega_{{\mathrm{{ADI}}}}/\omega={omega_adi(cfl)/omega:.4f}$"
    )
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=7, ncol=2)

for ax in axes[-1]:
    ax.set_xlabel(r"$t\,f_0$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$a(t)$")
fig.suptitle(
    rf"Forced-oscillator $a(t)$ with $\lambda\approx -k^2$ "
    rf"($N_z={NZ}$, $\omega=\frac{{3}}{{2}} ck$, $k\Delta x={kh:.3e}$)",
    fontsize=11,
)
fig.tight_layout()
plt.show()

print("q ≈ γ k² and ω_ADI/ω:")
for cfl in cfls_show:
    print(f"  CFL={cfl:g}: q={q_approx(cfl):.6f}, ω_ADI/ω={omega_adi(cfl)/omega:.6f}")


In [ ]:
# Overlay analytical a(t) with MMS simulation
# probe at x=L/4: sin(k L/4)=1 ⇒ E_probe ≡ a(t)
cfl_ov = 256.0

# coding labels → draft placements
sim_map = {
    "a": ("mid,1", "C0", a_mid1),
    "c": ("mid,2", "C2", a_mid2),
    "d": ("quarter", "C3", a_quarter),
}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (scheme, (pname, color, a_fn)) in zip(axes, sim_map.items()):
    out = results[scheme][cfl_ov]
    t_sim = out["t"]
    # use simulated final time (nsteps = ceil(...), so t_end ≥ N_PERIODS/f0)
    t_ov = np.linspace(0.0, float(t_sim[-1]), 2000)
    ax.plot(
        t_sim * f0,
        out["e_probe"],
        "x",
        color=color,
        ms=5,
        mew=1.2,
        label=f"sim {scheme.upper()} ({pname})",
    )
    ax.plot(t_ov * f0, a_fn(t_ov, cfl_ov), color=color, ls="--", lw=1.4, alpha=0.9, label=f"theory {pname}")
    ax.plot(t_ov * f0, np.cos(omega * t_ov), "k:", lw=1.0, alpha=0.6, label=r"MMS $\cos(\omega t)$")
    ax.set_title(rf"{pname}, CFL$={cfl_ov:g}$")
    ax.set_xlabel(r"$t\,f_0$")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=7)

axes[0].set_ylabel(r"$a(t)\equiv E(x=L/4)$")
fig.suptitle(
    rf"Theory ($\lambda\approx -k^2$) vs MMS sim, CFL$={cfl_ov:g}$, "
    rf"$q\approx{q_approx(cfl_ov):.3f}$, $N_z={NZ}$",
    fontsize=11,
)
fig.tight_layout()
plt.show()


